# 02 — Data Preprocessing & Preparation

This notebook preprocesses the satellite image dataset for CNN training:
1. **Resize** all images to 64×64
2. **Remove duplicates** from desert class
3. **Extract patches** from large desert images (256×256 → 16× 64×64)
4. **Balance** all classes to 1500 images
5. **Split** into train (80%) / test (20%) sets
6. **Normalize** pixel values
7. **Save** preprocessed dataset
8. **(Optional) Upsample** minority classes for augmented training data

In [1]:
import os
import hashlib
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import shutil
from tqdm import tqdm

# Reproducibility
random.seed(42)
np.random.seed(42)

# Paths
DATA_DIR = Path("../data")
PREP_DIR = Path("../data_preprocessed")
CLASSES = ["cloudy", "desert", "green_area", "water"]
CLASS_DIRS = {cls: DATA_DIR / cls for cls in CLASSES}

# Create preprocessed directory
PREP_DIR.mkdir(exist_ok=True)

print("Setup complete.")
print(f"  Original data dir: {DATA_DIR.resolve()}")
print(f"  Preprocessed data dir: {PREP_DIR.resolve()}")

Setup complete.
  Original data dir: /Users/michelangelonardi/Desktop/Università/Master/Bocconi Master/Year 2/Semester2 UW/AI for Agriculture/Final Project/AI_agri_project_noAI/data
  Preprocessed data dir: /Users/michelangelonardi/Desktop/Università/Master/Bocconi Master/Year 2/Semester2 UW/AI for Agriculture/Final Project/AI_agri_project_noAI/data_preprocessed


## 1. Load and Explore Dataset

In [2]:
# Load dataset metadata
records = []
for cls in CLASSES:
    paths = list(CLASS_DIRS[cls].glob("*.jpg")) + list(CLASS_DIRS[cls].glob("*.png"))
    for p in paths:
        with Image.open(p) as img:
            w, h = img.size
        records.append({"class": cls, "path": str(p), "filename": p.name, "width": w, "height": h})

df = pd.DataFrame(records)

print("=" * 60)
print("ORIGINAL DATASET")
print("=" * 60)
print(f"Total images: {len(df)}")
print()
print(df.groupby("class").agg({"filename": "count", "width": "min", "height": "min"}).rename(
    columns={"filename": "count", "width": "min_width", "height": "min_height"}).to_string())
print()
print("Image size breakdown by class:")
for cls in CLASSES:
    sizes = df[df["class"] == cls][["width", "height"]].drop_duplicates()
    print(f"  {cls:12s}: {len(sizes)} unique sizes")

ORIGINAL DATASET
Total images: 5631

            count  min_width  min_height
class                                   
cloudy       1500        256         256
desert       1131        256         256
green_area   1500         64          64
water        1500         64          64

Image size breakdown by class:
  cloudy      : 1 unique sizes
  desert      : 1 unique sizes
  green_area  : 1 unique sizes
  water       : 1 unique sizes


## 2. Remove Duplicates from Desert Class

In [3]:
# Identify and remove duplicates from desert class
def compute_md5(filepath, chunk_size=8192):
    """Compute MD5 hash of a file."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Hash only desert images
desert_hashes = defaultdict(list)
for _, row in df[df["class"] == "desert"].iterrows():
    h = compute_md5(row["path"])
    desert_hashes[h].append(row["path"])

# Find duplicates
duplicates = [files for files in desert_hashes.values() if len(files) > 1]
duplicate_paths = set()
for dup_group in duplicates:
    # Keep first, remove rest
    for path in dup_group[1:]:
        duplicate_paths.add(path)

print("=" * 60)
print("DUPLICATE REMOVAL (Desert Class)")
print("=" * 60)
print(f"Duplicate groups found: {len(duplicates)}")
print(f"Total duplicate files to remove: {len(duplicate_paths)}")

# Remove duplicates from dataset
df_dedup = df[~df["path"].isin(duplicate_paths)].copy()

print(f"\nBefore dedup: {len(df[df['class'] == 'desert'])} desert images")
print(f"After dedup:  {len(df_dedup[df_dedup['class'] == 'desert'])} desert images")
print(f"Dataset size: {len(df)} → {len(df_dedup)}")

DUPLICATE REMOVAL (Desert Class)
Duplicate groups found: 39
Total duplicate files to remove: 39

Before dedup: 1131 desert images
After dedup:  1092 desert images
Dataset size: 5631 → 5592


## 3. Resize & Extract Patches from Large Images

In [4]:
TARGET_SIZE = 64
PATCH_SIZE = 64

def extract_patches_64(image_array, patch_size=64):
    """
    Extract non-overlapping patches from an image.
    If image is 256x256, returns 16 patches of 64x64.
    """
    h, w = image_array.shape[:2]
    patches = []
    
    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            if y + patch_size <= h and x + patch_size <= w:
                patch = image_array[y:y+patch_size, x:x+patch_size]
                patches.append(patch)
    
    return patches

# Process all images: resize small ones, extract patches from large ones
processed_images = []  # List of (image_array, class, source_filename)

print("=" * 60)
print("PROCESSING IMAGES")
print("=" * 60)

for cls in CLASSES:
    cls_images = df_dedup[df_dedup["class"] == cls]
    patches_extracted = 0
    resized = 0
    
    for _, row in cls_images.iterrows():
        try:
            img = Image.open(row["path"]).convert("RGB")
            w, h = img.size
            
            # If image is 256x256 or larger, extract patches
            if w >= 256 and h >= 256:
                img_array = np.array(img)
                patches = extract_patches_64(img_array, patch_size=64)
                for patch in patches:
                    processed_images.append((patch, cls, row["filename"]))
                patches_extracted += len(patches)
            else:
                # Resize to 64x64
                img_resized = img.resize((TARGET_SIZE, TARGET_SIZE), Image.Resampling.LANCZOS)
                img_array = np.array(img_resized)
                processed_images.append((img_array, cls, row["filename"]))
                resized += 1
        except Exception as e:
            print(f"  Error processing {row['filename']}: {e}")
    
    print(f"{cls:12s}: {resized} resized, {patches_extracted} patches extracted → {resized + patches_extracted} total")

print()
print(f"Total processed images: {len(processed_images)}")

PROCESSING IMAGES
cloudy      : 0 resized, 24000 patches extracted → 24000 total
desert      : 0 resized, 17472 patches extracted → 17472 total
green_area  : 1500 resized, 0 patches extracted → 1500 total
water       : 1500 resized, 0 patches extracted → 1500 total

Total processed images: 44472


## 4. Balance Dataset to 1500 Images per Class

In [5]:
# Balance to 1500 images per class
TARGET_IMAGES_PER_CLASS = 1500
balanced_images = []

print("=" * 60)
print("BALANCING DATASET")
print("=" * 60)

for cls in CLASSES:
    cls_imgs = [(img, c, fn) for img, c, fn in processed_images if c == cls]
    
    if len(cls_imgs) >= TARGET_IMAGES_PER_CLASS:
        # Subsample if too many
        selected = random.sample(cls_imgs, TARGET_IMAGES_PER_CLASS)
        print(f"{cls:12s}: {len(cls_imgs)} → {TARGET_IMAGES_PER_CLASS} (subsampled)")
    else:
        # Use all if not enough
        selected = cls_imgs
        print(f"{cls:12s}: {len(cls_imgs)} (not enough for {TARGET_IMAGES_PER_CLASS})")
    
    balanced_images.extend(selected)

print()
print(f"Total balanced dataset: {len(balanced_images)} images")
print(f"Per class: {Counter([c for _, c, _ in balanced_images])}")

# Convert to array format
X = np.array([img for img, _, _ in balanced_images], dtype=np.float32)
y = np.array([cls for _, cls, _ in balanced_images])

print()
print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")

BALANCING DATASET
cloudy      : 24000 → 1500 (subsampled)
desert      : 17472 → 1500 (subsampled)
green_area  : 1500 → 1500 (subsampled)
water       : 1500 → 1500 (subsampled)

Total balanced dataset: 6000 images
Per class: Counter({'cloudy': 1500, 'desert': 1500, 'green_area': 1500, 'water': 1500})

Data shape: (6000, 64, 64, 3)
Labels shape: (6000,)


## 5. Train/Test Split (80/20)

In [6]:
# Stratified train/test split (80/20) to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("=" * 60)
print("TRAIN/TEST SPLIT (80/20)")
print("=" * 60)
print(f"Train set: {len(X_train)} images")
print(f"Test set:  {len(X_test)} images")
print()
print("Train set class distribution:")
for cls in CLASSES:
    count = np.sum(y_train == cls)
    pct = count / len(y_train) * 100
    print(f"  {cls:12s}: {count:4d} ({pct:5.1f}%)")

print()
print("Test set class distribution:")
for cls in CLASSES:
    count = np.sum(y_test == cls)
    pct = count / len(y_test) * 100
    print(f"  {cls:12s}: {count:4d} ({pct:5.1f}%)")

# Create numeric labels for classes
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

y_train_idx = np.array([class_to_idx[cls] for cls in y_train])
y_test_idx = np.array([class_to_idx[cls] for cls in y_test])

print()
print(f"Class mapping: {class_to_idx}")

TRAIN/TEST SPLIT (80/20)
Train set: 4800 images
Test set:  1200 images

Train set class distribution:
  cloudy      : 1200 ( 25.0%)
  desert      : 1200 ( 25.0%)
  green_area  : 1200 ( 25.0%)
  water       : 1200 ( 25.0%)

Test set class distribution:
  cloudy      :  300 ( 25.0%)
  desert      :  300 ( 25.0%)
  green_area  :  300 ( 25.0%)
  water       :  300 ( 25.0%)

Class mapping: {'cloudy': 0, 'desert': 1, 'green_area': 2, 'water': 3}


## 6. Apply Normalization (0-1)

In [7]:
# Normalize pixel values to [0, 1] range
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

print("=" * 60)
print("NORMALIZATION")
print("=" * 60)
print(f"Train set range: [{X_train_norm.min():.4f}, {X_train_norm.max():.4f}]")
print(f"Test set range:  [{X_test_norm.min():.4f}, {X_test_norm.max():.4f}]")
print()
print(f"Train mean (per channel):")
train_mean = X_train_norm.reshape(-1, 3).mean(axis=0)
print(f"  R: {train_mean[0]:.4f}, G: {train_mean[1]:.4f}, B: {train_mean[2]:.4f}")
print()
print(f"Train std (per channel):")
train_std = X_train_norm.reshape(-1, 3).std(axis=0)
print(f"  R: {train_std[0]:.4f}, G: {train_std[1]:.4f}, B: {train_std[2]:.4f}")
print()
print("✓ Data ready for CNN training")

NORMALIZATION
Train set range: [0.0000, 1.0000]
Test set range:  [0.0431, 1.0000]

Train mean (per channel):
  R: 0.3890, G: 0.4193, B: 0.4505

Train std (per channel):
  R: 0.2661, G: 0.1970, B: 0.1437

✓ Data ready for CNN training


## 7. Save Preprocessed Dataset

In [8]:
# Save preprocessed dataset
PREP_DIR.mkdir(exist_ok=True)

# Save as numpy arrays
np.save(PREP_DIR / "X_train.npy", X_train_norm)
np.save(PREP_DIR / "X_test.npy", X_test_norm)
np.save(PREP_DIR / "y_train.npy", y_train_idx)
np.save(PREP_DIR / "y_test.npy", y_test_idx)

# Save class mapping
import json
with open(PREP_DIR / "class_mapping.json", "w") as f:
    json.dump({"class_to_idx": class_to_idx, "idx_to_class": idx_to_class}, f)

print("=" * 60)
print("SAVED PREPROCESSED DATASET")
print("=" * 60)
print(f"Location: {PREP_DIR.resolve()}")
print()
print("Files saved:")
print(f"  ✓ X_train.npy  ({X_train_norm.nbytes / 1e6:.1f} MB)")
print(f"  ✓ X_test.npy   ({X_test_norm.nbytes / 1e6:.1f} MB)")
print(f"  ✓ y_train.npy  ({y_train_idx.nbytes / 1e3:.1f} KB)")
print(f"  ✓ y_test.npy   ({y_test_idx.nbytes / 1e3:.1f} KB)")
print(f"  ✓ class_mapping.json")
print()
print("Loading example:")
print("  X_train = np.load('data_preprocessed/X_train.npy')")
print("  y_train = np.load('data_preprocessed/y_train.npy')")

SAVED PREPROCESSED DATASET
Location: /Users/michelangelonardi/Desktop/Università/Master/Bocconi Master/Year 2/Semester2 UW/AI for Agriculture/Final Project/AI_agri_project_noAI/data_preprocessed

Files saved:
  ✓ X_train.npy  (235.9 MB)
  ✓ X_test.npy   (59.0 MB)
  ✓ y_train.npy  (38.4 KB)
  ✓ y_test.npy   (9.6 KB)
  ✓ class_mapping.json

Loading example:
  X_train = np.load('data_preprocessed/X_train.npy')
  y_train = np.load('data_preprocessed/y_train.npy')


In [15]:
X_train = np.load('../data_preprocessed/X_train.npy')
y_train = np.load('../data_preprocessed/y_train.npy')
X_train.shape, y_train.shape

((4800, 64, 64, 3), (4800,))

In [17]:
X_train[0, :, :, :], y_train[0]

(array([[[0.54901963, 0.5019608 , 0.42745098],
         [0.54901963, 0.49411765, 0.41960785],
         [0.5411765 , 0.4862745 , 0.4117647 ],
         ...,
         [0.45490196, 0.4392157 , 0.37254903],
         [0.45882353, 0.4392157 , 0.3764706 ],
         [0.45882353, 0.44313726, 0.38039216]],
 
        [[0.54901963, 0.5019608 , 0.42745098],
         [0.54901963, 0.49803922, 0.42352942],
         [0.5411765 , 0.49019608, 0.41568628],
         ...,
         [0.4627451 , 0.4392157 , 0.37254903],
         [0.4627451 , 0.4392157 , 0.3764706 ],
         [0.46666667, 0.44313726, 0.38039216]],
 
        [[0.54901963, 0.5019608 , 0.42745098],
         [0.54901963, 0.5019608 , 0.42745098],
         [0.54509807, 0.49411765, 0.41960785],
         ...,
         [0.46666667, 0.4392157 , 0.37254903],
         [0.47058824, 0.44313726, 0.3764706 ],
         [0.47058824, 0.44313726, 0.38039216]],
 
        ...,
 
        [[0.54901963, 0.5294118 , 0.45490196],
         [0.53333336, 0.5137255 , 0.43921

In [14]:
X_test = np.load('../data_preprocessed/X_test.npy')
y_test = np.load('../data_preprocessed/y_test.npy')
X_test.shape, y_test.shape

((1200, 64, 64, 3), (1200,))

## 8. (Optional) Create Upsampled Training Set for Augmentation

In [ ]:
"""
OPTIONAL: Create augmented training data with upsampling + augmentation.

Since all classes now have equal representation (1200 samples each in train set),
heavy upsampling is not strictly necessary. However, you can use augmentation
techniques to artificially increase dataset size and improve generalization.

Example approaches:
1. Random oversampling (repeat images)
2. Geometric augmentations (rotation, flips, crops)
3. Color augmentations (brightness, contrast, hue)
4. Mixed augmentations

This is useful for exploring how augmentation affects model performance.
"""

print("=" * 60)
print("UPSAMPLING & AUGMENTATION OPTIONS")
print("=" * 60)
print()
print("Current balanced train set: 1200 samples per class")
print()
print("Options for augmented training:")
print()
print("1. RANDOM OVERSAMPLING (via class weights in loss)")
print("   → No need to save; use class_weight in model.fit()")
print("   → Example: class_weight={0: 1, 1: 1, 2: 1, 3: 1}")
print()
print("2. DATA AUGMENTATION (in preprocessing pipeline)")
print("   → Use torchvision.transforms or albumentations")
print("   → Apply on-the-fly during training")
print("   → Recommended approach!")
print()
print("3. OFFLINE AUGMENTATION (pre-compute augmented images)")
print("   → Generate augmented copies and save")
print("   → Slower but reproducible")
print()
print("Recommended: Use on-the-fly augmentation during training")
print("(see CNN training notebook for examples)")
print()
print("Current dataset is balanced and ready for training ✓")